## Оглавление

- Введение: от подсчёта слов к их представлению
- Дистрибутивная гипотеза и векторное пространство
- Латентно-семантический анализ (LSA / LSI)
- Нейросетевая языковая модель Бенжио (NPLM)
- Word2Vec
- GloVe
- FastText
- Итоги раздела: какая мысль здесь развивалась
- Тематическое моделирование (Topic Modeling)
    - LSA как отправная точка
    - pLSA
    - LDA
    - NMF как альтернатива
    - Нейросетевое тематическое моделирование: Top2Vec и BERTopic
    - Что развивалось в этой ветви

---

К 1970 году компьютеры уже умели немало: искать документы по ключевым словам, считать частоты, предсказывать следующее слово по статистике коротких цепочек. Но для машины слова «кошка» и «кот» были так же далеки друг от друга, как «кошка» и «трактор»: две строки либо совпадают, либо нет, не существует промежуточного состояния, когда слова не сопадают, но "похожи". Дальнейшее развитие подтолкнуло к созданию такого представления, слово превратилось из просто "символьной метки" в точку непрерывного векторного пространства смыслов, в котором геометрическая близость отражает близость смысловую

## Векторная модель
Мы упоминали в первой главе, как ещё в 1950-х лингвисты Зеллиг Харрис и Джон Фёрс сформулировали принцип, позже озаглавленный "дистрибутивная гипотеза", который гласил: слово характеризуется контекстом, в которой оно употребляется. Если два слова систематически появляются в окружении одинъ и тех же слов, они обладают близкой семантикой. То есть "смысл" - это непрерывная величина, отюда название "дистрибутивная"

Прикладное воплощение эта мысль нашла в области информационного поиска. Напоминим, что информационный поиск - это дисциплина CS, предполагающая поиск релевантных запросу документов по базе документов. До 2000 поиск преимущественно по текстовому содержанию

В 1960–70-х Джерард Солтон с коллегами ввел термин векторная модель (vector space model): документ описывается вектором частот входящих в него слов, а близость документов измеряется углом между этими векторами. Множество документов составляют Document-Term матрицу, где по строкам отложены документы. а по столюцам слова

<img src="img/vector_space.png" width=350>

У такого представления есть несколько очевидных минусов: 
- такой вектор огромнен (только в английском языке насчитывается более 1 млн словоформ)
- он разрежен (почти полностью состоит из нулей)
- зашумлен (случайные слова )
- прямое сравнение строк не улавливает синонимию: два слова с близким смыслом могут почти не пересекаться по конкретным контекстам.

Проблему шума частично решает TF-IDF взвешивание

Позднее были предложены альтернативные способы взвешивания, например, BM25

Также можно сокращать размер словаря, но это купирует проблему, а не решает ее фундаментально.

Было бы хорошо иметь более компактное "сжатое" представление 

## Латентно-семантический анализ (LSA / LSI)
В конце 1980-х группа исследователей из Bellcore билась над практической проблемой поисковых систем: пользователь в запросе пишет "автомобиль", но документы со словом "машина" полностью игнорируются. В качестве решения предложили  способ извлечения семантики, который назвали латентно-семантическим анализом, он же латентно-семантическое индексирование — **LSA / LSI** [(Deerwester et al., 1990)](https://doi.org/10.1002/%28SICI%291097-4571%28199009%2941:6%3C391::AID-ASI1%3E3.0.CO;2-9). Оба названия синономичны, термин LSI (индексирование) прижился в информационном поиске, LSA (анализ) — в когнитивной науке и психолингвистике

Они взяли «Term-Document» матрицу $X$ размера $m \times n$, содержащую частоту каэждого из $m$ слов в каждом из $n$ документов (возможно взвешенные по TF-IDF) и применили к ней сингулярное разложение (SVD). Напомним, что SVD декомпозиция прямоугольной матрицы раскладывает ее в произведение 3х матриц, две из которых U и V ортогональные, а третья $\Sigma$ диагнональная:

$$X = U \Sigma V^{\top}$$

Замечательное свойство данного разложения, что диагональная матрица $\Sigma$ состоит из сингулярных значений, упорядоченных по убыванию. Это значит, что оставив только первые $k$ из них (обычно несколько сотен), мы получим приближенное усечённое разложение (Truncated SVD). Разложение - наилучшее в смысле нормы приближение оригиниальной матрицы $X$ матрицей ранга $k$:

$$X \approx X_k = U_k \Sigma_k V_k^{\top}$$

Нам в первую очередь важны матрицы U и V. Строки матрицы $U_k \Sigma_k$ можно рассматривать как $k$-мерные описания слов, а столбцы матрицы $\Sigma_k V_k^{\top}$ как векторы документов. Усечение — и есть ключевой ход: оно выбрасывает шум и заставляет слова, встречающиеся в похожих документах, получить близкие представления, даже если они ни разу не встретились в одном документе напрямую. 

Близость двух слов или документов при этом измеряют косинусной мерой их векторов $\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a}^{\top}\mathbf{b}}{\lVert \mathbf{a}\rVert\,\lVert \mathbf{b}\rVert}$. Так «автомобиль» и «машина» наконец окажутся рядом, поскольку что живут в похожих документах

Примеры практического применения подхода:
- первые попытки еще в 1960-ых применение факторного анализа для автоматической классификации документов (Borko & Bernick, 1963)
- в 1992 году LSI впервые применили в системе автоматического назначения рецензентов на научные статьи<br>нужно было сопоставить текстов с экспертами в данной области
- в 1994 году выдан патент на применение мультиязычного LSI: сопоставлять по смыслу написанных на разных языках
- в 1995 году LSI впервые задействовали для автоматической оценки сочинений, система измеряла семантическую близость студенческого текста с эталонным ответом
- К концу 1990-х метод стал повсеместно использоваться на практике

Методы LSA сделали шаг к . У модели LSA тоже есть важные ограничения
- к сожалению LSA описывает документы как «bag-of-words» модель - порядок слов никак не учитывается.
- плюс плохо справляется с многозначностью (все смыслы слова смешиваются в один вектор)
- требует пересчёта разложения при добавлении новых данных

## Первая нейросетевая языковая модель
В задаче языкового моделирования (предсказать следующее слово по предыдущим) до 2000 года безраздельно правили n-граммные модели. И у них была фундаментальная беда - то самое «проклятие размерности»:
- число возможных комбинаций из n слов астрономическое (`|V|ⁿ`), большинство из них ни разу не встречается в обучающих данных, соотвественно получает нулевую вероятность и ее приходится искусственно коректиовать<br><br>
- слова в таких моделях - дискретные символьные наборы, лишенные семантики («кот» и «собака» для них различаются ровно так же, как «кот» и «пылесос», просто разные индексы в словаре). Из-за этого модель не умеет нормально обобщать

Дабы решить эти две проблемы [(Benjio et al, 2003)](https://jmlr.org/papers/volume3/tmp/bengio03a.pdf) предложили принудительно сопоставлять каждому слову из словаря некоторый обучаемый вещественный вектор (порядка 30–100 измерений), который выполняет роль его представления. А вероятность каждого возможного продолжения оценивается двуслойной моделью

<img src="img/benjio.png" width=400> 

Фактически это был первый пример использования "обучаемых" эмбедингов, в том формате, какой мы знаем сейчас

 Это мгновенно "включило" обобщаемость: если в ходе обучения «кот» и «собака» получают близкие векторы, то модель, видевшая «кот сидел на полу», автоматически назначит разумную вероятность фразе «собака сидела на полу», даже если её в данных не было. Похожие слова → близкие векторы → переносимое знание. Вот так непрерывное пространство признаков лечит проклятие размерности.

Архитектурно это feed-forward нейросеть: векторы $n-1$ предыдущих слов $C(w_{t-n+1}), \dots, C(w_{t-1})$ конкатенируются в единый вектор $x$, проходят через скрытый слой с нелинейностью, а на выходе softmax по всему словарю $V$ даёт распределение вероятностей следующего слова:

$$\hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1}) = \frac{e^{y_{w_t}}}{\sum_{i \in V} e^{y_i}}, \qquad y = b + W x + U \tanh(d + H x).$$

Здесь $y_i$ — ненормированная оценка (logit) для $i$-го слова словаря, а матрицы $H, U, W$ и векторы смещений $b, d$ — обучаемые параметры сети. Обучается всё это максимизацией логарифма правдоподобия обучающего текста (с регуляризацией):

$$L = \frac{1}{T}\sum_{t} \log \hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

Тесты показали, что нейросестевая модель превзошла по метрике перплексии n-граммные язвковые модели больше чем на 10%. и главное, была доказана идея: обучаемые распределённые представления слов работают и побеждают детеминированные представления на частотах

Тем не менее у модели фундаментально есть пара узких мест:
- вычисление знаменателя для softmax - крайне дорогостоящая операция из-за суммы по всему словарю; она становится поблемой уже при словаре в десятки тысяч слов; в оригинальной работе авторы боролись с этим распараллеливанием подсчета; а в более поздних модификациях использовали приближенное вычисление через:
    - hierarchical softmax [(Bengio et al, 2005)](https://proceedings.mlr.press/r5/morin05a.html)
    - importance sampling<br><br>
- контекст фиксированной длины и небольшой<br>это сильно ограничивает выразительные возможности модели; для обхода ограничения прибегают к другому классу моделей - рекурсивные сети

#### Иерархический softmax
Идея иерархического softmax: параметризуем не слова, а внтуренние ноды дерева поиска (листья - слова). Каждое такое представление внутренней ноды перенаправляет $h$ влево или вправо. Таким образом для подсчета вероятности $P(w|h)$ нужно сделать $log(|V|)$ шагов. В модели Bengio дерево строили по WordStat

#### Importance Sampling 
sdfsdf

## Word2Vec
В 2013 году Томаш Миколов с коллегами из Google задался прагматичным вопросом: если нужны только векторы слов, зачем обучать полноценную языковую модель? Так появилось переосмсыление модели Бенжио, модель **Word2Vec** [(Mikolov et al., 2013)](https://arxiv.org/abs/1301.3781). Модель была намеренно упрощена: из неё выброшен дорогой скрытый нелинейный слой и осталась только линейная проекция, а языковая постановка уступила место . С этого момента термин эмбединг стал повсеместным и его начали массово использовать

Было предложено два симметричных варианта архитектуры: 
- CBOW (continuous bag-of-words) учит модель предсказывать центральное слово по окружающим его слева и справа словам
- skip-gram учит модель по центральному слову предсказывает окружающие

Оба подхода сопоставимы по качеству. Skip-gram обычно даёт лучшие представления для редких слов, но CBOW работает быстрее

Обратите внимание, что обе архитектуры используют контекст с обеих сторон от слова, а не только предшествующие слова, как языковая модель Бенжио

Формально skip-gram максимизирует среднюю логарифмическую вероятность контекстных слов при данном центральном слове $w_t$ в окне радиуса $c$:

$$\frac{1}{T}\sum_{t=1}^{T} \sum_{-c \le j \le c,\; j \ne 0} \log p(w_{t+j} \mid w_t),$$

где базовая (softmax) параметризация использует два набора векторов — «входной» $\mathbf{v}_w$ для центрального слова и «выходной» $\mathbf{u}_w$ для контекстного:

$$p(w_O \mid w_I) = \frac{\exp(\mathbf{u}_{w_O}^{\top}\mathbf{v}_{w_I})}{\sum_{w \in V}\exp(\mathbf{u}_{w}^{\top}\mathbf{v}_{w_I})}$$

Проблема вычисления знаменателя в Softmax никуда не делась. Mikolov предложил два пути решения:<Br>
а) уже знакомый иерархический softmax - параметризуем внутренние ноды дерева поиска. Здесбь отличие в том, что дерево предстваляет собой дерево Хаффмана - структуру, учитывающкю частоту слова<br>
б) в более поздней версии использовался negative sampling:  вместо нормировки по всему словарю модель учится отличать реальную пару «слово — контекст» от $k$ случайно подобранных «отрицательных» примеров $w_i \sim P_n(w)$:
$$\log \sigma(\mathbf{u}_{w_O}^{\top}\mathbf{v}_{w_I}) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)}\big[\log \sigma(-\mathbf{u}_{w_i}^{\top}\mathbf{v}_{w_I})\big],$$
где $\sigma(x) = 1/(1+e^{-x})$. Именно эти приёмы позволили обучать качественные векторы на миллиардах слов меньше чем за день.

Особенную популярность метод получил благодаря удивительному наблюдению - оказалось, что полученное пространство обладает линейной структурой: семантические и синтаксические отношения выражаются постоянными векторными сдвигами. Отсюда знаменитая арифметика аналогий — $\mathbf{v}_{\text{король}} - \mathbf{v}_{\text{мужчина}} + \mathbf{v}_{\text{женщина}} \approx \mathbf{v}_{\text{королева}}$

## GloVe
К 2014 году в области отчётливо сложились два лагеря: счётные факторизационные методы — наследники LSA, работающие с глобальной статистикой всей матрицы совместной встречаемости и предсказательные методы окна вроде Word2Vec, обучающиеся на локальных контекстах. 

Исследваотели из Стенфорда задались вопросом: нельзя ли взять лучшее от обоих? Так появился метод **GloVe** [(Pennington et al., 2014)](https://aclanthology.org/D14-1162/)

Строится глобальная матрица совместной встречаемости $X$, где $X_{ij}$ — сколько раз слово $j$ встречается в контексте слова $i$ по всему корпусу. Ключевое наблюдение авторов: осмысленную информацию несут не сами по себе совместные встречаемости, а их отношения. Отношение вероятностей $P_{ik}/P_{jk}$, где $P_{ik} = X_{ik}/\sum_l X_{il}$ — вероятность встретить пробное слово $k$ рядом со словом $i$, хорошо разделяет релевантные и нерелевантные ассоциации: оно велико, если $k$ ближе к $i$, чем к $j$, и мало в обратном случае. Из этого наблюдения выводится взвешенная задача наименьших квадратов, в которой скалярное произведение векторов приближает логарифм числа совместных встречаемостей:

$$J = \sum_{i,j=1}^{V} f(X_{ij})\,\big(\mathbf{w}_i^{\top}\tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\big)^2.$$

Здесь $\mathbf{w}_i$ и $\tilde{\mathbf{w}}_j$ — векторы слова и контекстного слова, $b_i, \tilde{b}_j$ — смещения, а весовая функция $f$ гасит вклад как слишком редких, так и чересчур частых пар:

$$f(x) = \begin{cases} (x/x_{\max})^{\alpha}, & x < x_{\max} \\ 1, & x \ge x_{\max} \end{cases}$$

с типичными $x_{\max} = 100$ и $\alpha = 3/4$.

Расчёт оправдался. GloVe обучается не проходом по тексту скользящим окном, а по уже собранной матрице статистик, что делает его эффективным и хорошо распараллеливаемым. По качеству на задачах аналогий и сходства слов он оказался сопоставим с Word2Vec — и на несколько лет пара Word2Vec / GloVe стала стандартным набором предобученных эмбеддингов, которыми инициализировали модели в огромном числе прикладных NLP-задач.

## FastText
К 2016 году Миколов уже работал в Facebook AI Research — и вместе с коллегами пересмотрели подход к Word2Vec и GloVe, которые присваивают каждому слову отдельный вектор и относятся к слову как к неделимому атому. Отсюда два следствия:
- игнорируется морфология: «бежать», «бежит», «убежал» получают разные векторы, хотя очевидно родственны, - это особенно критично для морфологически богатых языков, имеющих множество редких словоформ
- модели беспомощны перед новыми словами (out-of-vocabulary): для слова, не встретившегося при обучении, вектора просто нет

Ответом стал **FastText** [(Bojanowski et al., 2016)](https://arxiv.org/abs/1607.04606) — прямое развитие Word2Vec. Авторы берут архитектуру skip-gram, но меняют единицу представления: слово описывается не одним вектором, а как мешок символьных n-грамм. Слово с учётом граничных маркеров разбивается на подстроки из трёх–шести символов, и каждой такой n-грамме $g$ сопоставляется свой вектор $\mathbf{z}_g$. Оценка совместимости центрального слова $w$ (с множеством его n-грамм $\mathcal{G}_w$) и контекстного слова $c$ считается не как одно скалярное произведение, а как сумма по n-граммам:

$$s(w, c) = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g^{\top}\mathbf{v}_c.$$

Тем самым вектор целого слова — фактически сумма векторов входящих в него n-грамм; обучение же идёт по той же схеме предсказания контекста (с negative sampling), что и в Word2Vec.

Что это дало. Во-первых, модель улавливает морфологию «бесплатно»: слова с общими корнями и аффиксами разделяют символьные n-граммы, а значит, автоматически получают близкие векторы. Во-вторых — и это главное практическое преимущество — для любого нового слова, не встречавшегося при обучении, вектор можно собрать из его символьных n-грамм. Проблема out-of-vocabulary фактически снимается. При этом метод остаётся быстрым и обучается на больших корпусах так же эффективно, как Word2Vec.

На FastText «классическая» линия статических эмбеддингов, по сути, завершилась: идея дистрибутивных векторов доведена до уровня подслов, но слову по-прежнему соответствует один вектор независимо от контекста. Именно эта контекстная неизменность станет отправной точкой следующего этапа — контекстных представлений и трансформеров

## Тематическое моделирование (Topic Modeling)

Вернемся к LSA. Из той же матрицы совместной встречаемости выросла целая параллельная ветвь, задавшая другой вопрос. Если эмбеддинги спрашивают «как представить отдельное слово вектором», то тематические модели спрашивают «из каких скрытых тем состоит коллекция документов и как темы распределены по документам». Тема здесь — не метка, а распределение вероятностей по словам: скажем, тема «спорт» даёт высокую вероятность словам «матч», «команда», «гол». Документ же описывается как смесь нескольких тем в разных пропорциях.

Для нашего рассказа эта ветвь важна вдвойне. Во-первых, на ней отлично виден переход от чисто алгебраического сжатия (LSA) к явно вероятностным порождающим моделям. Во-вторых, её позднейшая эволюция (Top2Vec, BERTopic) напрямую использует эмбеддинги из основных глав — так две линии, счётная и предсказательная, в итоге сойдутся в одной задаче.

### LSA как отправная точка

Отсчитывать тематическое моделирование удобно от уже знакомого нам LSA. Он даёт «темы» неявно — как направления в пространстве, полученные из SVD. Беда в том, что эти направления не читаются как вероятности: в них встречаются отрицательные значения, и их нельзя интерпретировать в духе «тема на 30% состоит из таких-то слов». Следующие методы последовательно чинят именно это.

### pLSA

Первый шаг сделал в 1999 году Томас Хофманн, работавший тогда в Беркли. Его вероятностный латентно-семантический анализ — **pLSA** [(Hofmann, 1999)](https://arxiv.org/abs/1301.6705) — заменил алгебраическое разложение статистической моделью (первоисточник — статья в трудах конференции UAI 1999 года; по ссылке доступна её общедоступная arXiv-версия). Совместная встречаемость слова $w$ и документа $d$ объясняется через скрытую переменную — тему $z$: документ порождает тему с вероятностью $P(z \mid d)$, а тема порождает слово с вероятностью $P(w \mid z)$. Совместная вероятность раскладывается в сумму по темам:

$$P(w, d) = P(d)\sum_{z} P(z \mid d)\,P(w \mid z).$$

Обучение — подбор этих вероятностей методом максимального правдоподобия через EM-алгоритм. В результате, в отличие от LSA, каждая тема $P(w \mid z)$ — честное распределение по словам, которое можно прочитать и проинтерпретировать.

Но у модели Хофманна быстро обнаружилось ограничение, ставшее мотивацией следующего шага: pLSA моделирует только те документы, которые видел при обучении. У него нет порождающего механизма на уровне всей коллекции, число параметров растёт с числом документов, и он склонен к переобучению. По сути, pLSA неспособен приписать вероятность новому, ранее не виденному документу.

### LDA

Эту слабость взялись устранять Дэвид Блей, Эндрю Ын и Майкл Джордан. В 2003 году они представили латентное размещение Дирихле — **LDA** [(Blei et al., 2003)](https://www.jmlr.org/papers/v3/blei03a.html) — модель, которой предстояло стать эталоном жанра (работа вышла в JMLR, до эпохи arXiv).

Их ход: достроить pLSA до полноценной порождающей байесовской модели, добавив априорные распределения Дирихле над двумя вещами — над распределением тем в документе и над распределением слов в теме. Порождающая история для документа с параметрами $\alpha, \beta$ такова:

$$\theta_d \sim \text{Dir}(\alpha), \qquad z_{d,i} \sim \text{Cat}(\theta_d), \qquad w_{d,i} \sim \text{Cat}(\varphi_{z_{d,i}}), \qquad \varphi_k \sim \text{Dir}(\beta).$$

То есть для каждого документа сначала из распределения Дирихле берётся его смесь тем $\theta_d$; затем для каждого слова выбирается тема $z_{d,i}$ согласно этой смеси, а из соответствующего темового распределения $\varphi_k$ — конкретное слово. Априор Дирихле работает как регуляризатор: он «подталкивает» документы к небольшому числу выраженных тем, а темы — к небольшому числу характерных слов, что даёт более устойчивые и интерпретируемые результаты, чем pLSA.

Расчёт оправдался. LDA снимает главную слабость pLSA: это генеративная модель на уровне всей коллекции, поэтому она принципиально способна работать с новыми документами и меньше переобучается. Вывод параметров выполняется приближённо — вариационными методами или сэмплированием Гиббса. На полтора десятилетия LDA стал стандартом де-факто для анализа больших текстовых коллекций: тематизация новостных архивов, научных статей, обзоров товаров. Вокруг него выросло целое семейство расширений — коррелированные тематические модели (учитывающие, что темы могут коррелировать) и динамические (отслеживающие изменение тем во времени).

### NMF как альтернатива

Параллельно жила и более простая, линейно-алгебраическая альтернатива — неотрицательная матричная факторизация, **NMF**. В отличие от SVD в LSA, она раскладывает матрицу «слово — документ» $X$ на два множителя с неотрицательными элементами:

$$X \approx W H, \qquad W \ge 0,\; H \ge 0,$$

минимизируя $\lVert X - W H \rVert_F^2$ (или дивергенцию), где столбцы $W$ — темы (наборы весов слов), а столбцы $H$ — представления документов через темы. Именно ограничение неотрицательности делает результат интерпретируемым: документ представляется как сумма (а не разность) тематических компонент, а каждая тема — как неотрицательный набор весов слов. NMF часто оказывается быстрее LDA и на коротких текстах даёт сопоставимое качество, за что его любят как практичный инженерный вариант.

### Нейросетевое тематическое моделирование

К концу 2010-х линия тематического моделирования сомкнулась с линией эмбеддингов, за которой мы следили в основных главах. Новое поколение методов рассуждает так: зачем раскладывать матрицу частот, если можно работать прямо в пространстве плотных векторов, где семантически близкие документы уже лежат рядом?

Первым эту мысль оформил Димо Ангелов: его **Top2Vec** [(Angelov, 2020)](https://arxiv.org/abs/2008.09470) переосмысляет тему геометрически. Документы и слова совместно вкладываются в одно векторное пространство; затем кластеризацией находятся плотные скопления документов, центр каждого скопления объявляется вектором темы, а ближайшие к нему слова — её описанием. Практическое преимущество: метод сам определяет число тем и не требует стоп-слов, стемминга и лемматизации, поскольку опирается на семантику эмбеддингов, а не на «мешок слов».

Два года спустя Мартен Гроотендорст развил тот же принцип в модульный конвейер **BERTopic** [(Grootendorst, 2022)](https://arxiv.org/abs/2203.05794), ставший сегодня одним из самых популярных инструментов. Схема из четырёх шагов: сначала документы кодируются эмбеддингами из предобученного трансформера; затем размерность понижается (обычно UMAP); затем плотные кластеры выделяются алгоритмом HDBSCAN; наконец, для описания каждого кластера словами применяется c-TF-IDF — вариант TF-IDF, посчитанный не для отдельного документа, а для всего кластера $c$ как для единого «класса»:

$$W_{t,c} = \text{tf}_{t,c} \cdot \log\!\left(1 + \frac{A}{f_t}\right),$$

где $\text{tf}_{t,c}$ — частота слова $t$ в кластере $c$, $f_t$ — суммарная частота слова $t$ по всем кластерам, а $A$ — среднее число слов на кластер. Такая нормировка выделяет слова, характерные именно для этой темы. Модульность позволяет заменять любой из шагов независимо.

Для нашего рассказа эти два метода — красивая точка схождения: тематическое моделирование, начавшееся как факторизация счётной матрицы (LSA), в итоге стало надстройкой над нейросетевыми эмбеддингами. Формально Top2Vec и BERTopic выходят за верхнюю границу «2000–2015», но без них история этой ветви осталась бы оборванной.

### Что развивалось в этой ветви

Если основная линия раздела — путь от символа к вектору, то линия тематического моделирования — движение ко всё большей интерпретируемости и всё большей опоре на семантику. От неявных направлений LSA к явным вероятностным темам pLSA; от переобучающегося pLSA к устойчивой порождающей модели LDA; и, наконец, от «мешка слов» к плотным контекстным эмбеддингам в Top2Vec и BERTopic. Нить, связывающая её с остальным рассказом, всё та же: смысл извлекается из статистики контекстов, а прогресс состоит в том, чтобы делать эти представления богаче, устойчивее и ближе к семантике.